### Helpers

In [ ]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch.nn.functional as F

def _load_model_and_tokenizer(model_id, cache_dir=None):
    """Loads the model and tokenizer."""
    cwd = os.getcwd()
    cache_dir = cwd + "/huggingface/.cache"
    os.makedirs(cache_dir, exist_ok=True)

    print(f"Loading {model_id}...")
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        cache_dir=cache_dir,
        dtype=torch.float16,
        low_cpu_mem_usage=True
    )
    model.eval()
    return model, tokenizer

model_id = "qwen/qwen2.5-0.5B-instruct"
model, tokenizer = _load_model_and_tokenizer(model_id)

In [ ]:
from abc import ABC, abstractmethod
from typing import List

class Agent(ABC):
    @abstractmethod
    def query(self, prompt: str, labels: List[str]) -> List[float]:
        raise NotImplementedError("Subclasses must implement this method")

class HFAgent(Agent):
    SYSTEM_MESSAGE = "You are a helpful assistant. Answer shortly with only your choice with no explanation.\n\n"

    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer
        self.system_prompt = self.SYSTEM_MESSAGE
    
    def _convert_to_chat_template(self, text):
        messages = [
            {
                "role": "system",
                "content": self.system_prompt
            },
            {
                "role": "user",
                "content": text
            }
        ]
        return self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

q1_template = "You have two options:\nOption 1 - {A}\nOption 2 - {B}\nWhich do you prefer?\nAnswer: "
options = ['Green', 'Purple']
q1 = q1_template.format(A=options[0], B=options[1])
q1_rev = q1_template.format(A=options[1], B=options[0])

# Lotan v0

In [ ]:
class LotanHFAgentV0(HFAgent):
    def query(self, prompt: str, labels: List[str]) -> List[float]:
        prompt = self._convert_to_chat_template(prompt)
        scores = []
        metadata = {}
        for label in labels:
            full_text = prompt + " " + label
            input_ids = self.tokenizer.encode(full_text, return_tensors="pt").to(self.model.device)
            prompt_ids = self.tokenizer.encode(prompt, return_tensors="pt").to(self.model.device)
            prompt_len = prompt_ids.shape[1]
            with torch.no_grad():
                outputs = self.model(input_ids)
                logits = outputs.logits
            label_logits = logits[0, prompt_len-1 : -1, :] 
            label_ids = input_ids[0, prompt_len:]
            metadata[label] = {
                'input_ids': input_ids,
                'logits': logits,
                'label_ids': label_ids
            }
            loss = F.cross_entropy(label_logits, label_ids, reduction='sum')
            scores.append(-loss.item())
        return scores, metadata

In [ ]:
lotan_v0 = LotanHFAgentV0(model, tokenizer)
lotan_v0_scores, lotan_v0_metadata = lotan_v0.query(q1, labels=['Option 1', 'Option 2'])
lotan_v0_scores

In [ ]:
print(tokenizer.decode(lotan_v0_metadata['Option 1']['input_ids'][0]))

In [ ]:
lotan_v0_metadata['Option 1']

In [ ]:
tokenizer.decode(lotan_v0_metadata['Option 1']['label_ids'])

This space is the problem!

# Itay v0

In [ ]:
class ItayHFAgentV0(HFAgent):
    """ The original model from Itay's paper """
    def get_chat_format_one_side(self, text, role):
        return {
            "role": role,
            "content": text,
        }

    def convert_to_chat_format(self, text, few_shots_texts=None):
        """
        Uses the apply_chat_template function in the tokenizer of the predictor to convert the text to chat format
        """
        SYSTEM_MESSAGE = "You are a helpful assistant. Answer shortly with only your choice with no explanation.\n\n"
        # if there are no few shots, just add the system message to the text
        if few_shots_texts is None:
            messages = [
                self.get_chat_format_one_side(SYSTEM_MESSAGE + text, "user"),
            ]
        # if there are few shots, add the system message to the first shot

        else:
            # the first shot needs the system message before the text
            messages = [
                self.get_chat_format_one_side(
                    SYSTEM_MESSAGE + few_shots_texts[0]["question"], "user"
                ),
                self.get_chat_format_one_side(
                    few_shots_texts[0]["answer"], "assistant"
                ),
            ]

            # after that, add all the other shot in the user and assistent format
            for shot in few_shots_texts[1:]:
                messages.extend(
                    [
                        self.get_chat_format_one_side(shot["question"], "user"),
                        self.get_chat_format_one_side(shot["answer"], "assistant"),
                    ]
                )

            # finaly, add the actual text to the user
            messages.append(self.get_chat_format_one_side(text, "user"))

        prompt = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,  # , return_tensors="pt"
        )

        return prompt
    
    def query(self, prompt, labels):
        prompt = [self.convert_to_chat_format(p) for p in prompt]
        # concat labels to the corrposnded input text
        input_with_answers = [i + label for label in labels for i in prompt]
        # get labels tokens ids
        labels_tokens = self.tokenizer(labels, add_special_tokens=False)["input_ids"]
        # get the last token id of each label
        labels_tokens = [label[-1] for label in labels_tokens]
        # Ensure pad token exists before padding
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        # Get encodings for each input text using direct call instead of batch_encode_plus
        input_enc = self.tokenizer(
            input_with_answers,
            return_tensors="pt",
            padding="longest",
        )
        for k, v in input_enc.items():
            input_enc[k] = v.to(self.model.device)

        # Get model output logits
        model_output = self.model(**input_enc)

        # Compute the log probabilities associated with each of the labels
        labels_log_probs = F.log_softmax(model_output.logits, dim=-1)

        # Get the ids of the token before the last token before padding (to see the probablity of the last token given the one before the last token)
        before_padding_ids = (
            input_enc["input_ids"].ne(self.tokenizer.pad_token_id).sum(-1) - 2
        )

        # Collect labels scores from the -2 token in labels_log_probs (the one that predict the last token)
        # and collect for each line the id in labels_tokens
        labels_scores = labels_log_probs[:, before_padding_ids, labels_tokens]

        # Need just the diagonal of the matrix, as it the prob of the label for each line
        labels_scores = torch.diag(labels_scores)

        metadata = {
            'input_ids': input_enc.input_ids,
            'logits': model_output.logits,
        }

        return labels_scores, metadata


In [ ]:
itay_v0 = ItayHFAgentV0(model, tokenizer)
itay_v0_scores, itay_v0_metadata = itay_v0.query([q1], labels=['Option 1','Option 2'])
itay_v0_scores

# Lotan v1

In [ ]:
class LotanHFAgentV1(HFAgent):
    def query(self, prompt: str, labels: List[str]) -> List[float]:
        prompt = self._convert_to_chat_template(prompt)
        scores = []
        metadata = {}
        prompt_ids = self.tokenizer.encode(prompt, return_tensors="pt").to(self.model.device)

        for label in labels:
            label_ids = self.tokenizer.encode(label, return_tensors="pt").to(self.model.device)
            input_ids = torch.cat([prompt_ids, label_ids], dim=1).to(self.model.device)
            prompt_len = prompt_ids.shape[1]
            with torch.no_grad():
                outputs = self.model(input_ids)
                logits = outputs.logits
            label_logits = logits[0, prompt_len-1 : -1, :] 
            label_ids = input_ids[0, prompt_len:]
            metadata[label] = {
                'input_ids': input_ids,
                'logits': logits,
                'label_ids': label_ids
            }
            loss = F.cross_entropy(label_logits, label_ids, reduction='sum')
            scores.append(-loss.item())
        return scores, metadata

In [ ]:
lotan_v1 = LotanHFAgentV1(model, tokenizer)
lotan_v1_scores, lotan_v1_metadata = lotan_v1.query(q1, labels=['Option 1', 'Option 2'])
lotan_v1_scores

In [ ]:
tokenizer.decode(lotan_v1_metadata['Option 1']['label_ids'])

lotan_v1 calculates:
`Score = log P("Option" | prompt) + log P(" 1" | prompt + "Option")`

while itay_v0 calculates:
`Score = log P(" 1" | prompt + "Option")`

This adds a fixed "bias" for lotan_v1 equal `log P("Option" | prompt)`

In [ ]:
tokenizer.decode(lotan_v1_metadata['Option 1']['input_ids'][0])

# Itay v1

In [ ]:
class ItayHFAgentV1(HFAgent):
    """
    - Fixed system prompt issue.
    - Used simpler convert_to_chat_format
    - covnerting prompts in the query function.
    """
    def query(self, prompts, labels):
        if not isinstance(prompts, list):
            prompts = [prompts]
        prompts = [self._convert_to_chat_template(p) for p in prompts]
        # concat labels to the corrposnded input text
        input_with_answers = [i + label for label in labels for i in prompts]
        # get labels tokens ids
        labels_tokens = self.tokenizer(labels, add_special_tokens=False)["input_ids"]
        # get the last token id of each label
        labels_tokens = [label[-1] for label in labels_tokens]
        # Ensure pad token exists before padding
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        # Get encodings for each input text using direct call instead of batch_encode_plus
        input_enc = self.tokenizer(
            input_with_answers,
            return_tensors="pt",
            padding="longest",
        )
        for k, v in input_enc.items():
            input_enc[k] = v.to(self.model.device)

        # Get model output logits
        model_output = self.model(**input_enc)

        # Compute the log probabilities associated with each of the labels
        labels_log_probs = F.log_softmax(model_output.logits, dim=-1)

        # Get the ids of the token before the last token before padding (to see the probablity of the last token given the one before the last token)
        before_padding_ids = (
            input_enc["input_ids"].ne(self.tokenizer.pad_token_id).sum(-1) - 2
        )

        # Collect labels scores from the -2 token in labels_log_probs (the one that predict the last token)
        # and collect for each line the id in labels_tokens
        labels_scores = labels_log_probs[:, before_padding_ids, labels_tokens]

        # Need just the diagonal of the matrix, as it the prob of the label for each line
        labels_scores = torch.diag(labels_scores)

        metadata = {
            'input_ids': input_enc.input_ids,
            'logits': model_output.logits,
        }

        return labels_scores, metadata


In [ ]:
itay_v1 = ItayHFAgentV1(model, tokenizer)
itay_v1_scores, itay_v1_metadata = itay_v1.query([q1], labels=['Option 1', 'Option 2'])
itay_v1_scores

In [ ]:
print(tokenizer.decode(itay_v1_metadata['input_ids'][0]))

In [ ]:
q2_template = "Which do you prefer: {A} or {B}?"
q2 = q2_template.format(A=options[0], B=options[1])

In [ ]:
itay_v1_scores_2, itay_v1_metadata_2 = itay_v1.query([q1], labels=options)
itay_v1_scores_2

# Bradley-Terry

In [1]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize
from scipy.special import expit  # This is the sigmoid function

def compute_probabilities(df: pd.DataFrame) -> pd.DataFrame:
    """Converts raw logprobs into softmax probabilities."""
    df = df.copy()
    # expit(x) is numerically stable sigmoid
    df['prob_a'] = expit(df['score_a'] - df['score_b'])
    df['prob_b'] = 1 - df['prob_a']
    return df

def aggregate_to_combinations(df: pd.DataFrame) -> pd.DataFrame:
    """Standardizes pairs to (i, j) where i < j, and averages their probabilities."""
    records = []
    for _, row in df.iterrows():
        # Lexicographical sort to ensure consistent (item_i, item_j) ordering
        if row['option_a'] < row['option_b']:
            i, j = row['option_a'], row['option_b']
            p_i = row['prob_a']
        else:
            i, j = row['option_b'], row['option_a']
            p_i = row['prob_b'] # If order was flipped, we track prob of the new 'i'
            
        records.append({'item_i': i, 'item_j': j, 'prob_i_over_j': p_i})
        
    comb_df = pd.DataFrame(records)
    # Average the probabilities for every unique pair combination
    return comb_df.groupby(['item_i', 'item_j'], as_index=False).mean()

def create_item_mapping(combinations_df: pd.DataFrame) -> dict:
    """Maps unique string items to integer indices 0 to N-1."""
    items = pd.concat([combinations_df['item_i'], combinations_df['item_j']]).unique()
    return {item: idx for idx, item in enumerate(sorted(items))}

def fit_bradley_terry(combinations_df: pd.DataFrame, item_mapping: dict, alpha: float = 1e-4) -> np.ndarray:
    """Fits the Bradley-Terry model using Scipy."""
    n_items = len(item_mapping)
    
    # Map item strings to their array indices
    i_indices = combinations_df['item_i'].map(item_mapping).values
    j_indices = combinations_df['item_j'].map(item_mapping).values
    targets = combinations_df['prob_i_over_j'].values
    
    def objective(r):
        """BCE Loss function to minimize."""
        # Calculate predicted probabilities: sigmoid(r_i - r_j)
        logits = r[i_indices] - r[j_indices]
        preds = expit(logits)
        
        # Binary Cross-Entropy
        eps = 1e-15 # Prevent log(0)
        bce = -targets * np.log(preds + eps) - (1 - targets) * np.log(1 - preds + eps)
        
        # L2 Regularization (Ridge)
        # Because BT scores are translation invariant (r_i - r_j is the same if we add +5 to both),
        # we need a tiny penalty to anchor the scores around 0 so the optimizer doesn't drift.
        l2_penalty = alpha * np.sum(r**2)
        
        return np.sum(bce) + l2_penalty

    # Initialize all latent scores to 0
    initial_r = np.zeros(n_items)
    
    # Run the L-BFGS-B optimizer
    result = minimize(objective, initial_r, method='L-BFGS-B')
    
    if not result.success:
        print(f"Warning: Optimization failed. {result.message}")
        
    return result.x # Returns the array of fitted latent scores        

def get_predicted_probabilities(latent_scores: np.ndarray, combinations_df: pd.DataFrame, item_mapping: dict) -> np.ndarray:
    """Generates the model's predicted probabilities for the combinations."""
    i_indices = combinations_df['item_i'].map(item_mapping).values
    j_indices = combinations_df['item_j'].map(item_mapping).values
    
    logits = latent_scores[i_indices] - latent_scores[j_indices]
    return expit(logits)

def evaluate_fit(combinations_df: pd.DataFrame, predicted_probs: np.ndarray) -> dict:
    """Calculates MSE and Residual Deviance."""
    targets = combinations_df['prob_i_over_j'].values
    
    mse = np.mean((targets - predicted_probs)**2)
    
    # Residual Deviance (KL divergence scaled by 2)
    eps = 1e-15
    deviance = 2 * np.sum(
        targets * np.log((targets + eps) / (predicted_probs + eps)) +
        (1 - targets) * np.log((1 - targets + eps) / (1 - predicted_probs + eps))
    )
    
    return {'MSE': mse, 'Deviance': deviance}

In [2]:
df = pd.read_csv("data/qwen-0.5-foods-20260512_163106/scores.csv")
df.head()

,Unnamed: 0,template,option_a,option_b,prompt,score_a,score_b
0,0,You have two options:\nOption 1 - {A}\nOption ...,Pizza,Sushi,You have two options:\nOption 1 - Pizza\nOptio...,-0.879883,-0.536133
1,1,You have two options:\nOption 1 - {A}\nOption ...,Pizza,Burger,You have two options:\nOption 1 - Pizza\nOptio...,-0.757812,-0.632812
2,2,You have two options:\nOption 1 - {A}\nOption ...,Pizza,Pasta,You have two options:\nOption 1 - Pizza\nOptio...,-1.267578,-0.330566
3,3,You have two options:\nOption 1 - {A}\nOption ...,Pizza,Salad,You have two options:\nOption 1 - Pizza\nOptio...,-0.879883,-0.536133
4,4,You have two options:\nOption 1 - {A}\nOption ...,Pizza,Steak,You have two options:\nOption 1 - Pizza\nOptio...,-0.852539,-0.555664


In [3]:
df_probs = compute_probabilities(df)
df_probs.head()

,Unnamed: 0,template,option_a,option_b,prompt,score_a,score_b,prob_a,prob_b
0,0,You have two options:\nOption 1 - {A}\nOption ...,Pizza,Sushi,You have two options:\nOption 1 - Pizza\nOptio...,-0.879883,-0.536133,0.414899,0.585101
1,1,You have two options:\nOption 1 - {A}\nOption ...,Pizza,Burger,You have two options:\nOption 1 - Pizza\nOptio...,-0.757812,-0.632812,0.468791,0.531209
2,2,You have two options:\nOption 1 - {A}\nOption ...,Pizza,Pasta,You have two options:\nOption 1 - Pizza\nOptio...,-1.267578,-0.330566,0.281504,0.718496
3,3,You have two options:\nOption 1 - {A}\nOption ...,Pizza,Salad,You have two options:\nOption 1 - Pizza\nOptio...,-0.879883,-0.536133,0.414899,0.585101
4,4,You have two options:\nOption 1 - {A}\nOption ...,Pizza,Steak,You have two options:\nOption 1 - Pizza\nOptio...,-0.852539,-0.555664,0.426322,0.573678


In [4]:
# Validate compute_probabilities

df['score_a_exp'] = np.exp(df['score_a'])
df['score_b_exp'] = np.exp(df['score_b'])
df['prob_a'] = df['score_a_exp'] / (df['score_a_exp'] + df['score_b_exp'])
df['prob_b'] = df['score_b_exp'] / (df['score_a_exp'] + df['score_b_exp'])
joined = df.join(df_probs, lsuffix='_raw', rsuffix='_computed')
joined[['option_a_raw', 'option_b_raw', 'prob_a_raw', 'prob_b_raw', 'prob_a_computed', 'prob_b_computed']].head()
joined['prob_a_sub'] = joined['prob_a_raw'] - joined['prob_a_computed']
joined['prob_b_sub'] = joined['prob_b_raw'] - joined['prob_b_computed']
joined[['prob_a_sub', 'prob_b_sub']].describe()

,prob_a_sub,prob_b_sub
count,1.806000e+03,1.806000e+03
mean,-3.396446e-18,3.972766e-18
std,5.422754e-17,5.839717e-17
min,-2.220446e-16,-1.387779e-16
25%,0.000000e+00,0.000000e+00
50%,0.000000e+00,0.000000e+00
75%,0.000000e+00,0.000000e+00
max,1.110223e-16,1.526557e-16


In [5]:
df_combos = aggregate_to_combinations(df_probs)
df_combos.head()

,item_i,item_j,prob_i_over_j
0,Burger,Pasta,0.448254
1,Burger,Pizza,0.470925
2,Burger,Salad,0.452947
3,Burger,Steak,0.495707
4,Burger,Sushi,0.453101


In [6]:
mapping = create_item_mapping(df_combos)
mapping

{'Burger': 0,
 'Pasta': 1,
 'Pizza': 2,
 'Salad': 3,
 'Steak': 4,
 'Sushi': 5,
 'Tacos': 6}

In [7]:
fitted_scores = fit_bradley_terry(df_combos, mapping)
fitted_scores

array([-0.1525259 ,  0.05102452, -0.03038546,  0.04170498, -0.19953558,
        0.07596338,  0.21375374])

In [8]:
df_probs.rename(columns={'option_a': 'item_i', 'option_b': 'item_j'}, inplace=True)
df_probs.rename(columns={'score_a': 'score_i', 'score_b': 'score_j'}, inplace=True)
df_probs.rename(columns={'prob_a': 'prob_i', 'prob_b': 'prob_j'}, inplace=True)
df_probs['prob_i_over_j'] = df_probs['prob_i'] / (df_probs['prob_i'] + df_probs['prob_j'])

fitted_scores_probs = fit_bradley_terry(df_probs, mapping)
fitted_scores_probs

array([-0.15254351,  0.05103014, -0.03038888,  0.04170972, -0.19955887,
        0.07597185,  0.21377843])

In [9]:
diff_bt_scores = np.linalg.norm(fitted_scores_probs - fitted_scores)
diff_bt_scores

np.float64(3.999383023291731e-05)

In [14]:
df_probs

,Unnamed: 0,template,item_i,item_j,prompt,score_i,score_j,prob_i,prob_j,prob_i_over_j
0,0,You have two options:\nOption 1 - {A}\nOption ...,Pizza,Sushi,You have two options:\nOption 1 - Pizza\nOptio...,-0.879883,-0.536133,0.414899,0.585101,0.414899
1,1,You have two options:\nOption 1 - {A}\nOption ...,Pizza,Burger,You have two options:\nOption 1 - Pizza\nOptio...,-0.757812,-0.632812,0.468791,0.531209,0.468791
2,2,You have two options:\nOption 1 - {A}\nOption ...,Pizza,Pasta,You have two options:\nOption 1 - Pizza\nOptio...,-1.267578,-0.330566,0.281504,0.718496,0.281504
3,3,You have two options:\nOption 1 - {A}\nOption ...,Pizza,Salad,You have two options:\nOption 1 - Pizza\nOptio...,-0.879883,-0.536133,0.414899,0.585101,0.414899
4,4,You have two options:\nOption 1 - {A}\nOption ...,Pizza,Steak,You have two options:\nOption 1 - Pizza\nOptio...,-0.852539,-0.555664,0.426322,0.573678,0.426322
...,...,...,...,...,...,...,...,...,...,...
1801,1801,You have two options:\nOption 1 - {A}\nOption ...,Tacos,Sushi,You have two options:\nOption 1 - Tacos\nOptio...,-0.259033,-1.477539,0.771801,0.228199,0.771801
1802,1802,You have two options:\nOption 1 - {A}\nOption ...,Tacos,Burger,You have two options:\nOption 1 - Tacos\nOptio...,-0.241699,-1.539062,0.785391,0.214609,0.785391
1803,1803,You have two options:\nOption 1 - {A}\nOption ...,Tacos,Pasta,You have two options:\nOption 1 - Tacos\nOptio...,-0.259033,-1.477539,0.771801,0.228199,0.771801
1804,1804,You have two options:\nOption 1 - {A}\nOption ...,Tacos,Salad,You have two options:\nOption 1 - Tacos\nOptio...,-0.248535,-1.513672,0.779909,0.220091,0.779909


In [20]:
df

,Unnamed: 0,template,option_a,option_b,prompt,score_a,score_b,score_a_exp,score_b_exp,prob_a,prob_b
0,0,You have two options:\nOption 1 - {A}\nOption ...,Pizza,Sushi,You have two options:\nOption 1 - Pizza\nOptio...,-0.879883,-0.536133,0.414832,0.585006,0.414899,0.585101
1,1,You have two options:\nOption 1 - {A}\nOption ...,Pizza,Burger,You have two options:\nOption 1 - Pizza\nOptio...,-0.757812,-0.632812,0.468691,0.531096,0.468791,0.531209
2,2,You have two options:\nOption 1 - {A}\nOption ...,Pizza,Pasta,You have two options:\nOption 1 - Pizza\nOptio...,-1.267578,-0.330566,0.281513,0.718517,0.281504,0.718496
3,3,You have two options:\nOption 1 - {A}\nOption ...,Pizza,Salad,You have two options:\nOption 1 - Pizza\nOptio...,-0.879883,-0.536133,0.414832,0.585006,0.414899,0.585101
4,4,You have two options:\nOption 1 - {A}\nOption ...,Pizza,Steak,You have two options:\nOption 1 - Pizza\nOptio...,-0.852539,-0.555664,0.426331,0.573691,0.426322,0.573678
...,...,...,...,...,...,...,...,...,...,...,...
1801,1801,You have two options:\nOption 1 - {A}\nOption ...,Tacos,Sushi,You have two options:\nOption 1 - Tacos\nOptio...,-0.259033,-1.477539,0.771797,0.228199,0.771801,0.228199
1802,1802,You have two options:\nOption 1 - {A}\nOption ...,Tacos,Burger,You have two options:\nOption 1 - Tacos\nOptio...,-0.241699,-1.539062,0.785292,0.214582,0.785391,0.214609
1803,1803,You have two options:\nOption 1 - {A}\nOption ...,Tacos,Pasta,You have two options:\nOption 1 - Tacos\nOptio...,-0.259033,-1.477539,0.771797,0.228199,0.771801,0.228199
1804,1804,You have two options:\nOption 1 - {A}\nOption ...,Tacos,Salad,You have two options:\nOption 1 - Tacos\nOptio...,-0.248535,-1.513672,0.779942,0.220100,0.779909,0.220091


In [ ]:
from src.pref_models import fit_bradley_terry as fit_bt

df['winner'] = df_probs.apply(lambda row: row['item_i'] if row['prob_i'] > row['prob_j'] else row['item_j'], axis=1)
df.rename(columns={'option_a': 'item_a', 'option_b': 'item_b'}, inplace=True)
fit_bt(df_probs, mapping.keys())

(     Item  BT_Score
 0  Burger -0.619558
 1   Pasta  0.097139
 2   Pizza -0.158176
 3   Salad  0.175406
 4   Steak -0.720333
 5   Sushi  0.304566
 6   Tacos  0.920956,
 array([[ 0., 24., 34., 25., 41., 27., 22.],
        [62.,  0., 47., 40., 54., 41., 28.],
        [52., 39.,  0., 34., 55., 36., 20.],
        [61., 46., 52.,  0., 63., 36., 25.],
        [45., 32., 31., 23.,  0., 18., 11.],
        [59., 45., 50., 50., 68.,  0., 29.],
        [64., 58., 66., 61., 75., 57.,  0.]]))

In [ ]:
predicted_probs = get_predicted_probabilities(fitted_scores, df_combos, mapping)
predicted_probs

array([0.44928737, 0.46950279, 0.45159436, 0.51175026, 0.44312491,
       0.4094403 , 0.52034126, 0.50232987, 0.56231436, 0.49376561,
       0.45940723, 0.48198519, 0.54218699, 0.47343782, 0.43926656,
       0.56001934, 0.49143624, 0.4570936 , 0.43155761, 0.39812367,
       0.46560681])

In [ ]:
metrics = evaluate_fit(df_combos, predicted_probs)
metrics

{'MSE': np.float64(3.575345985137255e-05),
 'Deviance': np.float64(0.0030217008816874583)}

In [ ]:
final_rankings = {item: fitted_scores[idx] for item, idx in mapping.items()}
# Sort descending
print(sorted(final_rankings.items(), key=lambda x: x[1], reverse=True))

[('Tacos', np.float64(0.2137537438549655)), ('Sushi', np.float64(0.07596337908410732)), ('Pasta', np.float64(0.05102451920103756)), ('Salad', np.float64(0.041704981180214915)), ('Pizza', np.float64(-0.030385455180150945)), ('Burger', np.float64(-0.15252590175349204)), ('Steak', np.float64(-0.19953557870150865))]
